# Website A/B Testing - Lab

## Introduction

In this lab, you'll get another chance to practice your skills at conducting a full A/B test analysis. It will also be a chance to practice your data exploration and processing skills! The scenario you'll be investigating is data collected from the homepage of a music app page for audacity.

## Objectives

You will be able to:
* Analyze the data from a website A/B test to draw relevant conclusions
* Explore and analyze web action data

## Exploratory Analysis

Start by loading in the dataset stored in the file 'homepage_actions.csv'. Then conduct an exploratory analysis to get familiar with the data.

> Hints:
    * Start investigating the id column:
        * How many viewers also clicked?
        * Are there any anomalies with the data; did anyone click who didn't view?
        * Is there any overlap between the control and experiment groups?
            * If so, how do you plan to account for this in your experimental design?

In [2]:
#Your code here
import pandas as pd

# Load the dataset
df = pd.read_csv('homepage_actions.csv')

# Display the first few rows to understand the data structure
display(df.head())

# Investigate the 'id' column and check for anomalies
print("\nUnique ids:", df['id'].nunique())
print("Total rows:", len(df))

# Check if anyone clicked without viewing
clicked_not_viewed = df[(df['action'] == 'click') & (df['action'] == 'view')].empty
print("\nDid anyone click who didn't view?:", not clicked_not_viewed)


# Check overlap between control and experiment groups
control_ids = df[df['group'] == 'control']['id'].unique()
experiment_ids = df[df['group'] == 'experiment']['id'].unique()
overlap_ids = set(control_ids).intersection(set(experiment_ids))

print("\nNumber of users in control group:", len(control_ids))
print("Number of users in experiment group:", len(experiment_ids))
print("Number of users in both groups (overlap):", len(overlap_ids))

# How many viewers also clicked?
# First find users who viewed
viewers = df[df['action'] == 'view']['id'].unique()

# Then find users who clicked
clickers = df[df['action'] == 'click']['id'].unique()

# Find the intersection of viewers and clickers
viewers_who_clicked = set(viewers).intersection(set(clickers))

print("\nNumber of viewers who also clicked:", len(viewers_who_clicked))

,timestamp,id,group,action
0,2016-09-24 17:42:27.839496,804196,experiment,view
1,2016-09-24 19:19:03.542569,434745,experiment,view
2,2016-09-24 19:36:00.944135,507599,experiment,view
3,2016-09-24 19:59:02.646620,671993,control,view
4,2016-09-24 20:26:14.466886,536734,experiment,view



Unique ids: 6328
Total rows: 8188

Did anyone click who didn't view?: False

Number of users in control group: 3332
Number of users in experiment group: 2996
Number of users in both groups (overlap): 0

Number of viewers who also clicked: 1860


## Conduct a Statistical Test

Conduct a statistical test to determine whether the experimental homepage was more effective than that of the control group.

In [3]:
import statsmodels.api as sm
import numpy as np

# Create a new dataframe with one row per user and a binary click column
# First, get all unique user ids
user_ids = df['id'].unique()

# Create an empty list to store user data
user_data = []

# Iterate through each user id
for user_id in user_ids:
    # Get all actions for the current user
    user_actions = df[df['id'] == user_id]

    # Determine the group (assuming each user is only in one group based on previous check)
    group = user_actions['group'].iloc[0]

    # Check if the user clicked (at least one 'click' action)
    clicked = int('click' in user_actions['action'].values)

    # Append the user data to the list
    user_data.append({'id': user_id, 'group': group, 'clicked': clicked})

# Create a new DataFrame from the list
user_df = pd.DataFrame(user_data)

# Calculate the number of views (users) and clicks for each group
control_views = user_df[user_df['group'] == 'control'].shape[0]
control_clicks = user_df[(user_df['group'] == 'control') & (user_df['clicked'] == 1)].shape[0]

experiment_views = user_df[user_df['group'] == 'experiment'].shape[0]
experiment_clicks = user_df[(user_df['group'] == 'experiment') & (user_df['clicked'] == 1)].shape[0]

print("Control Group:")
print(f"Views: {control_views}, Clicks: {control_clicks}")
print("\nExperiment Group:")
print(f"Views: {experiment_views}, Clicks: {experiment_clicks}")

# Perform a z-test for proportions
# Count of successes (clicks) and trials (views) for each group
count = np.array([control_clicks, experiment_clicks])
nobs = np.array([control_views, experiment_views])

# Perform the z-test
z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='smaller') # alternative='smaller' because we hypothesize the experiment group has a higher click-through rate

print("\nZ-test for Proportions:")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpret the results
alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject the null hypothesis. The experiment group has a statistically significantly higher click-through rate than the control group.")
else:
    print("\nResult: Fail to reject the null hypothesis. There is no statistically significant difference in click-through rates between the groups.")

Control Group:
Views: 3332, Clicks: 932

Experiment Group:
Views: 2996, Clicks: 928

Z-test for Proportions:
Z-statistic: -2.6186
P-value: 0.0044

Result: Reject the null hypothesis. The experiment group has a statistically significantly higher click-through rate than the control group.


## Verifying Results

One sensible formulation of the data to answer the hypothesis test above would be to create a binary variable representing each individual in the experiment and control group. This binary variable would represent whether or not that individual clicked on the homepage; 1 for they did and 0 if they did not.

The variance for the number of successes in a sample of a binomial variable with n observations is given by:

## $n\bullet p (1-p)$

Given this, perform 3 steps to verify the results of your statistical test:
1. Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group.
2. Calculate the number of standard deviations that the actual number of clicks was from this estimate.
3. Finally, calculate a p-value using the normal distribution based on this z-score.

### Step 1:
Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group.

In [4]:
# Calculate the click-through rate of the control group
control_ctr = control_clicks / control_views

# Calculate the expected number of clicks in the experiment group
expected_experiment_clicks = experiment_views * control_ctr

print(f"Expected number of clicks in the experiment group (based on control CTR): {expected_experiment_clicks:.2f}")

Expected number of clicks in the experiment group (based on control CTR): 838.02


### Step 2:
Calculate the number of standard deviations that the actual number of clicks was from this estimate.

In [5]:
# Calculate the variance for the number of clicks in the experiment group under the null hypothesis
# The null hypothesis is that the experiment group has the same click-through rate as the control group
null_hypothesis_ctr = control_clicks / control_views
variance_experiment_clicks = experiment_views * null_hypothesis_ctr * (1 - null_hypothesis_ctr)

# Calculate the standard deviation
std_dev_experiment_clicks = np.sqrt(variance_experiment_clicks)

# Calculate the difference between the actual and expected clicks
difference_in_clicks = experiment_clicks - expected_experiment_clicks

# Calculate the z-score
z_score = difference_in_clicks / std_dev_experiment_clicks

print(f"Standard deviation of expected clicks in experiment group: {std_dev_experiment_clicks:.2f}")
print(f"Difference between actual and expected clicks: {difference_in_clicks:.2f}")
print(f"Z-score: {z_score:.4f}")

Standard deviation of expected clicks in experiment group: 24.57
Difference between actual and expected clicks: 89.98
Z-score: 3.6625


### Step 3:
Finally, calculate a p-value using the normal distribution based on this z-score.

In [8]:
from scipy.stats import norm

# P-value for a right-tailed test (experiment CTR > control CTR)
p_value_manual = 1 - norm.cdf(z_score)

print(f"P-value calculated from z-score: {p_value_manual:.4f}")

P-value calculated from z-score: 0.0001


### Analysis:

Does this result roughly match that of the previous statistical test?

> Comment: These two p-values do not roughly match. The manually calculated p-value is significantly smaller than the one from the initial statistical test.

## Summary

In this lab, you continued to get more practice designing and conducting AB tests. This required additional work preprocessing and formulating the initial problem in a suitable manner. Additionally, you also saw how to verify results, strengthening your knowledge of binomial variables, and reviewing initial statistical concepts of the central limit theorem, standard deviation, z-scores, and their accompanying p-values.